# Ablation Study Analysis — Multimodal World Model

This notebook provides a comprehensive analysis of all ablation experiments comparing architectural components, text content variants, difficulty scaling, and parameter-matched baselines on the **distract_cheetah_run** environment.

**Ablation groups:**
- **A — Component Isolation:** Measures contribution of FiLM, TextGate, and CLIP text
- **B — Text Content:** Tests semantic understanding (nonsense / adversarial text)
- **F — Difficulty Sweep:** CNN vs. Multimodal across easy / medium / hard distractors
- **H — Parameter Matching:** Wider CNN baseline to control for model capacity

In [8]:
import json
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.15)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.facecolor": "white",
})

print("Libraries loaded.")

Libraries loaded.


## 1. Load Data

Load metrics from `metrics.jsonl` files for all 12 ablation runs.

In [9]:
BASE_LOGDIR = Path("/nfs-stor/salem.lahlou/asharma/logdir/ablations")
TASK = "distract_cheetah_run"

# Ablation registry: id -> (directory name, display label, group)
ABLATIONS = {
    "A5": ("ablation_a5_cnn_baseline",        "A5: CNN Baseline",             "component"),
    "A1": ("ablation_a1_random_text",          "A1: Random Text",              "component"),
    "A2": ("ablation_a2_film_only",            "A2: FiLM Only",                "component"),
    "A3": ("ablation_a3_gate_only",            "A3: Gate Only",                "component"),
    "A4": ("ablation_a4_full_multimodal",      "A4: Full Multimodal",          "component"),
    "B3": ("ablation_b3_nonsense_text",        "B3: Nonsense Text",            "text_content"),
    "B6": ("ablation_b6_adversarial_text",     "B6: Adversarial Text",         "text_content"),
    "F1_CNN_med":  ("ablation_f1_cnn_medium",          "F1: CNN Medium",       "difficulty"),
    "F1_CNN_hard": ("ablation_f1_cnn_hard",            "F1: CNN Hard",         "difficulty"),
    "F1_MM_med":   ("ablation_f1_multimodal_medium",   "F1: Multimodal Medium","difficulty"),
    "F1_MM_hard":  ("ablation_f1_multimodal_hard",     "F1: Multimodal Hard",  "difficulty"),
    "H3": ("ablation_h3_wider_cnn",            "H3: Wider CNN (~5M)",          "parameter"),
}

# Architecture metadata
ARCH_META = {
    "A5": {"FiLM": False, "TextGate": False, "Text": "None"},
    "A1": {"FiLM": True,  "TextGate": True,  "Text": "Random"},
    "A2": {"FiLM": True,  "TextGate": False, "Text": "Real CLIP"},
    "A3": {"FiLM": False, "TextGate": True,  "Text": "Real CLIP"},
    "A4": {"FiLM": True,  "TextGate": True,  "Text": "Real CLIP"},
    "B3": {"FiLM": True,  "TextGate": True,  "Text": "Nonsense"},
    "B6": {"FiLM": True,  "TextGate": True,  "Text": "Adversarial"},
    "H3": {"FiLM": False, "TextGate": False, "Text": "None"},
}


def load_metrics(logdir):
    """Load metrics.jsonl into a list of dicts."""
    path = logdir / "metrics.jsonl"
    if not path.exists():
        return []
    records = []
    with open(path) as f:
        for line in f:
            records.append(json.loads(line))
    return records


def extract_series(records, key):
    """Extract (steps, values) for a given metric key."""
    steps, vals = [], []
    for r in records:
        if key in r:
            steps.append(r["step"])
            vals.append(r[key])
    return np.array(steps), np.array(vals)


# Load all data
all_metrics = {}
for ab_id, (dirname, label, group) in ABLATIONS.items():
    logdir = BASE_LOGDIR / dirname / TASK
    all_metrics[ab_id] = load_metrics(logdir)
    n = len(all_metrics[ab_id])
    print(f"  {ab_id:15s} -> {n:5d} records  ({label})")

print(f"\nLoaded {len(all_metrics)} ablation runs.")

  A5              ->  1311 records  (A5: CNN Baseline)
  A1              ->  1311 records  (A1: Random Text)
  A2              ->  1311 records  (A2: FiLM Only)
  A3              ->  1311 records  (A3: Gate Only)
  A4              ->  1311 records  (A4: Full Multimodal)
  B3              ->  1311 records  (B3: Nonsense Text)
  B6              ->  1311 records  (B6: Adversarial Text)
  F1_CNN_med      ->     0 records  (F1: CNN Medium)
  F1_CNN_hard     ->     0 records  (F1: CNN Hard)
  F1_MM_med       ->  1251 records  (F1: Multimodal Medium)
  F1_MM_hard      ->  1311 records  (F1: Multimodal Hard)
  H3              ->   857 records  (H3: Wider CNN (~5M))

Loaded 12 ablation runs.


In [10]:
# Helper functions

def smooth(values, weight=0.9):
    """Exponential moving average."""
    s = np.zeros_like(values, dtype=float)
    s[0] = values[0]
    for i in range(1, len(values)):
        s[i] = weight * s[i - 1] + (1 - weight) * values[i]
    return s


def final_score(records, key="episode/eval_score", last_n=5):
    """Mean and std of the last N evaluation scores."""
    steps, vals = extract_series(records, key)
    if len(vals) == 0:
        return np.nan, np.nan
    n = min(last_n, len(vals))
    return float(np.mean(vals[-n:])), float(np.std(vals[-n:]))


# Build summary dataframe
rows = []
for ab_id, (dirname, label, group) in ABLATIONS.items():
    mean, std = final_score(all_metrics[ab_id])
    row = {"Ablation": ab_id, "Label": label, "Group": group,
           "Mean Score": mean, "Std Score": std}
    if ab_id in ARCH_META:
        row.update(ARCH_META[ab_id])
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df = summary_df.set_index("Ablation")
summary_df.style.format({"Mean Score": "{:.1f}", "Std Score": "{:.1f}"}).background_gradient(
    subset=["Mean Score"], cmap="RdYlGn"
)

ImportError: Pandas requires version '3.1.2' or newer of 'jinja2' (version '3.0.3' currently installed).

---
## 2. Training Curves — All Ablations

Eval score over training steps for every ablation. Smoothed with EMA (weight=0.9) and raw values shown as transparent background.

In [ ]:
# Color palette for ablations
COLORS = {
    "A5": "#d62728",  # red
    "A1": "#ff7f0e",  # orange
    "A2": "#2ca02c",  # green
    "A3": "#9467bd",  # purple
    "A4": "#1f77b4",  # blue
    "B3": "#8c564b",  # brown
    "B6": "#e377c2",  # pink
    "H3": "#bcbd22",  # olive
    "F1_CNN_med":  "#d62728",
    "F1_CNN_hard": "#d62728",
    "F1_MM_med":   "#1f77b4",
    "F1_MM_hard":  "#1f77b4",
}

LINESTYLES = {
    "A5": "-", "A1": "--", "A2": "--", "A3": "--", "A4": "-",
    "B3": ":", "B6": ":", "H3": "-.",
    "F1_CNN_med": "--", "F1_CNN_hard": ":", "F1_MM_med": "--", "F1_MM_hard": ":",
}

# Plot all training curves on one figure
fig, ax = plt.subplots(figsize=(14, 7))

# Plot main ablations (exclude F1 difficulty variants for clarity)
main_ids = ["A5", "A1", "A2", "A3", "A4", "B3", "B6", "H3"]
for ab_id in main_ids:
    steps, vals = extract_series(all_metrics[ab_id], "episode/eval_score")
    if len(steps) == 0:
        continue
    label = ABLATIONS[ab_id][1]
    ax.plot(steps, smooth(vals), color=COLORS[ab_id], linestyle=LINESTYLES[ab_id],
            linewidth=2.2, label=label)
    ax.plot(steps, vals, color=COLORS[ab_id], alpha=0.12, linewidth=0.6)

ax.set_xlabel("Environment Steps")
ax.set_ylabel("Eval Score")
ax.set_title("Training Curves — All Ablations (distract_cheetah_run)")
ax.legend(loc="lower right", ncol=2, framealpha=0.9)
ax.set_xlim(left=0)
plt.tight_layout()
plt.show()

---
## 3. Component Isolation (A-series)

Bar chart comparing final performance when systematically removing FiLM, TextGate, or replacing CLIP text with random vectors.

| Ablation | FiLM | TextGate | Text Source | Tests |
|----------|------|----------|-------------|-------|
| A5 — CNN Baseline | No | No | None | Pure vision reference |
| A1 — Random Text | Yes | Yes | Random vector | Architecture vs. semantics |
| A3 — Gate Only | No | Yes | Real CLIP | Gate alone sufficient? |
| A2 — FiLM Only | Yes | No | Real CLIP | FiLM alone sufficient? |
| A4 — Full Multimodal | Yes | Yes | Real CLIP | Complete system |

In [ ]:
component_ids = ["A5", "A1", "A3", "A2", "A4"]
component_labels = ["CNN Baseline\n(A5)", "Random Text\n(A1)", "Gate Only\n(A3)",
                     "FiLM Only\n(A2)", "Full Multimodal\n(A4)"]
component_colors = ["#d62728", "#ff7f0e", "#9467bd", "#2ca02c", "#1f77b4"]

means = [final_score(all_metrics[ab])[0] for ab in component_ids]
stds  = [final_score(all_metrics[ab])[1] for ab in component_ids]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.3, 1]})

# --- Bar chart ---
ax = axes[0]
bars = ax.bar(range(len(means)), means, yerr=stds, color=component_colors,
              edgecolor="black", linewidth=0.8, capsize=5, zorder=3)
ax.set_xticks(range(len(means)))
ax.set_xticklabels(component_labels)
ax.set_ylabel("Final Eval Score (last 5 evals)")
ax.set_title("Component Isolation — Final Score")

# Value labels
for bar, m in zip(bars, means):
    if not np.isnan(m):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f"{m:.0f}", ha="center", va="bottom", fontsize=11, fontweight="bold")

# Highlight best
best_idx = int(np.nanargmax(means))
bars[best_idx].set_edgecolor("#FFD700")
bars[best_idx].set_linewidth(2.5)

# --- Training curves side-by-side ---
ax2 = axes[1]
for ab_id, color in zip(component_ids, component_colors):
    steps, vals = extract_series(all_metrics[ab_id], "episode/eval_score")
    if len(steps) == 0:
        continue
    ls = "-" if ab_id in ("A5", "A4") else "--"
    lw = 2.5 if ab_id in ("A5", "A4") else 1.8
    ax2.plot(steps, smooth(vals), color=color, linestyle=ls, linewidth=lw,
             label=ABLATIONS[ab_id][1])
    ax2.plot(steps, vals, color=color, alpha=0.1, linewidth=0.5)

ax2.set_xlabel("Environment Steps")
ax2.set_ylabel("Eval Score")
ax2.set_title("Component Isolation — Training Curves")
ax2.legend(fontsize=9, loc="lower right")
ax2.set_xlim(left=0)

plt.tight_layout()
plt.show()

---
## 4. Text Content Ablation (B-series)

Does the model use semantic meaning from text? Compare real task descriptions vs. nonsense (shuffled words) vs. adversarial (opposite instructions) vs. no text at all.

In [ ]:
text_ids = ["B6", "B3", "A5", "A4"]
text_labels = ["Adversarial\n(B6)", "Nonsense\n(B3)", "CNN Baseline\n(A5, no text)", "Real Text\n(A4, ours)"]
text_colors = ["#d62728", "#ff7f0e", "#7f7f7f", "#1f77b4"]

t_means = [final_score(all_metrics[ab])[0] for ab in text_ids]
t_stds  = [final_score(all_metrics[ab])[1] for ab in text_ids]

fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.2, 1]})

# --- Bar chart ---
ax = axes[0]
bars = ax.bar(range(len(t_means)), t_means, yerr=t_stds, color=text_colors,
              edgecolor="black", linewidth=0.8, capsize=5, zorder=3)
ax.set_xticks(range(len(t_means)))
ax.set_xticklabels(text_labels)
ax.set_ylabel("Final Eval Score")
ax.set_title("Text Content Ablation — Does Semantics Matter?")

for bar, m in zip(bars, t_means):
    if not np.isnan(m):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f"{m:.0f}", ha="center", va="bottom", fontsize=11, fontweight="bold")

# --- Training curves ---
ax2 = axes[1]
for ab_id, color in zip(text_ids, text_colors):
    steps, vals = extract_series(all_metrics[ab_id], "episode/eval_score")
    if len(steps) == 0:
        continue
    ls = "-" if ab_id in ("A4", "A5") else "--"
    ax2.plot(steps, smooth(vals), color=color, linestyle=ls, linewidth=2.2,
             label=ABLATIONS[ab_id][1])
    ax2.plot(steps, vals, color=color, alpha=0.1, linewidth=0.5)

ax2.set_xlabel("Environment Steps")
ax2.set_ylabel("Eval Score")
ax2.set_title("Text Content — Training Curves")
ax2.legend(fontsize=9, loc="lower right")
ax2.set_xlim(left=0)

plt.tight_layout()
plt.show()

---
## 5. Difficulty Sweep (F-series)

How do CNN and Multimodal scale as distractor difficulty increases from easy to hard?

In [ ]:
difficulties = ["Easy", "Medium", "Hard"]
cnn_ids = ["A5", "F1_CNN_med", "F1_CNN_hard"]
mm_ids  = ["A4", "F1_MM_med",  "F1_MM_hard"]

cnn_means = [final_score(all_metrics[ab])[0] for ab in cnn_ids]
cnn_stds  = [final_score(all_metrics[ab])[1] for ab in cnn_ids]
mm_means  = [final_score(all_metrics[ab])[0] for ab in mm_ids]
mm_stds   = [final_score(all_metrics[ab])[1] for ab in mm_ids]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Line plot ---
ax = axes[0]
x = np.arange(len(difficulties))
ax.errorbar(x, cnn_means, yerr=cnn_stds, marker="o", markersize=9,
            linewidth=2.5, capsize=6, color="#d62728", label="CNN Baseline")
ax.errorbar(x, mm_means, yerr=mm_stds, marker="s", markersize=9,
            linewidth=2.5, capsize=6, color="#1f77b4", label="Multimodal (Ours)")
ax.fill_between(x, cnn_means, mm_means, alpha=0.12, color="#1f77b4",
                label="Multimodal advantage")

ax.set_xticks(x)
ax.set_xticklabels(difficulties)
ax.set_xlabel("Distractor Difficulty")
ax.set_ylabel("Final Eval Score")
ax.set_title("Difficulty Sweep — CNN vs. Multimodal")
ax.legend(fontsize=10)

# Annotate gap at each difficulty
for i in range(len(difficulties)):
    gap = mm_means[i] - cnn_means[i]
    mid = (mm_means[i] + cnn_means[i]) / 2
    if not np.isnan(gap):
        ax.annotate(f"+{gap:.0f}", xy=(i + 0.1, mid), fontsize=10,
                    fontweight="bold", color="#1f77b4")

# --- Training curves for all difficulty variants ---
ax2 = axes[1]
diff_runs = [
    ("A5",          "CNN Easy",       "#d62728", "-",  2.0),
    ("F1_CNN_med",  "CNN Medium",     "#d62728", "--", 1.5),
    ("F1_CNN_hard", "CNN Hard",       "#d62728", ":",  1.5),
    ("A4",          "MM Easy",        "#1f77b4", "-",  2.0),
    ("F1_MM_med",   "MM Medium",      "#1f77b4", "--", 1.5),
    ("F1_MM_hard",  "MM Hard",        "#1f77b4", ":",  1.5),
]

for ab_id, label, color, ls, lw in diff_runs:
    steps, vals = extract_series(all_metrics[ab_id], "episode/eval_score")
    if len(steps) == 0:
        continue
    ax2.plot(steps, smooth(vals), color=color, linestyle=ls, linewidth=lw, label=label)

ax2.set_xlabel("Environment Steps")
ax2.set_ylabel("Eval Score")
ax2.set_title("Difficulty Sweep — Training Curves")
ax2.legend(fontsize=8, ncol=2, loc="lower right")
ax2.set_xlim(left=0)

plt.tight_layout()
plt.show()

---
## 6. Parameter-Matched Comparison (H-series)

Is multimodal's advantage due to text semantics or just having more parameters? The wider CNN (H3, depth=77, ~5M params) matches the multimodal encoder's parameter count.

In [ ]:
param_ids = ["A5", "H3", "A4"]
param_labels = ["CNN Baseline\n(A5, ~220K)", "Wider CNN\n(H3, ~5M)", "Multimodal\n(A4, ~5M)"]
param_colors = ["#d62728", "#ff7f0e", "#1f77b4"]

p_means = [final_score(all_metrics[ab])[0] for ab in param_ids]
p_stds  = [final_score(all_metrics[ab])[1] for ab in param_ids]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1, 1]})

# --- Bar chart ---
ax = axes[0]
bars = ax.bar(range(len(p_means)), p_means, yerr=p_stds, color=param_colors,
              edgecolor="black", linewidth=0.8, capsize=5, zorder=3, width=0.6)
ax.set_xticks(range(len(p_means)))
ax.set_xticklabels(param_labels)
ax.set_ylabel("Final Eval Score")
ax.set_title("Parameter-Matched Comparison")

for bar, m in zip(bars, p_means):
    if not np.isnan(m):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f"{m:.0f}", ha="center", va="bottom", fontsize=12, fontweight="bold")

# Bracket annotation showing "same params"
ax.annotate("", xy=(1, max(p_means[1], p_means[2]) * 1.08),
            xytext=(2, max(p_means[1], p_means[2]) * 1.08),
            arrowprops=dict(arrowstyle="<->", color="gray", lw=1.5))
ax.text(1.5, max(p_means[1], p_means[2]) * 1.10, "~same params",
        ha="center", fontsize=9, color="gray")

# --- Training curves ---
ax2 = axes[1]
for ab_id, color in zip(param_ids, param_colors):
    steps, vals = extract_series(all_metrics[ab_id], "episode/eval_score")
    if len(steps) == 0:
        continue
    ax2.plot(steps, smooth(vals), color=color, linewidth=2.5, label=ABLATIONS[ab_id][1])
    ax2.plot(steps, vals, color=color, alpha=0.1, linewidth=0.5)

ax2.set_xlabel("Environment Steps")
ax2.set_ylabel("Eval Score")
ax2.set_title("Parameter-Matched — Training Curves")
ax2.legend(fontsize=10, loc="lower right")
ax2.set_xlim(left=0)

plt.tight_layout()
plt.show()

---
## 7. Text Gate Analysis

How does the learned text gate value evolve during training? The gate controls how much text is mixed into the RSSM input (sigmoid(-3) ~ 4.7% at initialization). Comparing gate dynamics across ablations reveals whether and how the model learns to use text.

In [ ]:
# Ablations that have gate metrics
gate_ids = ["A1", "A3", "A4", "B3", "B6"]
gate_colors = {"A1": "#ff7f0e", "A3": "#9467bd", "A4": "#1f77b4",
               "B3": "#8c564b", "B6": "#e377c2"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- (a) Gate mean over training ---
ax = axes[0]
for ab_id in gate_ids:
    steps, vals = extract_series(all_metrics[ab_id], "train/encoder/text_gate_mean")
    if len(steps) == 0:
        continue
    ax.plot(steps, smooth(vals, 0.92), color=gate_colors[ab_id], linewidth=2,
            label=ABLATIONS[ab_id][1])
    ax.plot(steps, vals, color=gate_colors[ab_id], alpha=0.1, linewidth=0.5)

ax.axhline(y=0.047, color="gray", linestyle="--", alpha=0.6, linewidth=1)
ax.text(0.02, 0.047 + 0.005, "init (sigmoid(-3))", transform=ax.get_yaxis_transform(),
        fontsize=8, color="gray")
ax.set_xlabel("Environment Steps")
ax.set_ylabel("Text Gate Mean")
ax.set_title("(a) Text Gate Value Over Training")
ax.legend(fontsize=8, loc="best")
ax.set_xlim(left=0)

# --- (b) Gate std over training ---
ax = axes[1]
for ab_id in gate_ids:
    steps, vals = extract_series(all_metrics[ab_id], "train/encoder/text_gate_std")
    if len(steps) == 0:
        continue
    ax.plot(steps, smooth(vals, 0.92), color=gate_colors[ab_id], linewidth=2,
            label=ABLATIONS[ab_id][1])

ax.set_xlabel("Environment Steps")
ax.set_ylabel("Text Gate Std")
ax.set_title("(b) Text Gate Variance Over Training")
ax.legend(fontsize=8, loc="best")
ax.set_xlim(left=0)

# --- (c) Final gate value vs final score (scatter) ---
ax = axes[2]
for ab_id in gate_ids:
    steps_g, vals_g = extract_series(all_metrics[ab_id], "train/encoder/text_gate_mean")
    score_mean, _ = final_score(all_metrics[ab_id])
    if len(vals_g) == 0 or np.isnan(score_mean):
        continue
    final_gate = float(np.mean(vals_g[-10:]))
    ax.scatter(final_gate, score_mean, color=gate_colors[ab_id], s=120,
               edgecolors="black", linewidth=0.8, zorder=5)
    ax.annotate(ABLATIONS[ab_id][1], (final_gate, score_mean),
                textcoords="offset points", xytext=(8, 5), fontsize=9)

ax.set_xlabel("Final Gate Value (mean of last 10)")
ax.set_ylabel("Final Eval Score")
ax.set_title("(c) Gate Value vs. Performance")

plt.tight_layout()
plt.show()

---
## 8. Training Loss Comparison

Compare key training losses across ablations to understand how different architectures affect world model learning.

In [ ]:
loss_keys = ["train/loss/dyn", "train/loss/rep", "train/loss/rew", "train/loss/barlow"]
loss_titles = ["Dynamics Loss", "Representation Loss", "Reward Loss", "Barlow Loss"]
plot_ids = ["A5", "A4", "A1", "B3", "B6", "H3"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, key, title in zip(axes.flat, loss_keys, loss_titles):
    for ab_id in plot_ids:
        steps, vals = extract_series(all_metrics[ab_id], key)
        if len(steps) == 0:
            continue
        ax.plot(steps, smooth(vals, 0.92), color=COLORS[ab_id], linewidth=1.8,
                label=ABLATIONS[ab_id][1],
                linestyle=LINESTYLES.get(ab_id, "-"))

    ax.set_xlabel("Environment Steps")
    ax.set_ylabel("Loss")
    ax.set_title(title)
    ax.legend(fontsize=8, loc="best")
    ax.set_xlim(left=0)

plt.tight_layout()
plt.show()

---
## 9. Policy & Value Diagnostics

Compare policy-side metrics (returns, advantage, entropy) to understand how different encoders affect downstream RL learning.

In [ ]:
policy_keys = ["train/ret", "train/adv", "train/action_entropy", "train/val"]
policy_titles = ["Imagined Returns", "Advantage", "Action Entropy", "Value Estimate"]
plot_ids_policy = ["A5", "A4", "A1", "H3"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, key, title in zip(axes.flat, policy_keys, policy_titles):
    for ab_id in plot_ids_policy:
        steps, vals = extract_series(all_metrics[ab_id], key)
        if len(steps) == 0:
            continue
        ax.plot(steps, smooth(vals, 0.92), color=COLORS[ab_id], linewidth=2,
                label=ABLATIONS[ab_id][1])

    ax.set_xlabel("Environment Steps")
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.legend(fontsize=9, loc="best")
    ax.set_xlim(left=0)

plt.tight_layout()
plt.show()

---
## 10. Pairwise Improvement Heatmap

Relative improvement (%) of each ablation over every other ablation. Positive values (green) mean the row ablation outperforms the column ablation.

In [ ]:
heatmap_ids = ["A5", "A1", "A3", "A2", "B3", "B6", "H3", "A4"]
heatmap_labels = [ABLATIONS[ab][1] for ab in heatmap_ids]
heatmap_scores = [final_score(all_metrics[ab])[0] for ab in heatmap_ids]

n = len(heatmap_ids)
improvement = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        if heatmap_scores[j] != 0 and not np.isnan(heatmap_scores[i]) and not np.isnan(heatmap_scores[j]):
            improvement[i, j] = ((heatmap_scores[i] - heatmap_scores[j]) / abs(heatmap_scores[j])) * 100

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.eye(n, dtype=bool)

sns.heatmap(improvement, annot=True, fmt=".1f", cmap="RdYlGn", center=0,
            xticklabels=heatmap_labels, yticklabels=heatmap_labels,
            mask=mask, linewidths=0.5, ax=ax,
            cbar_kws={"label": "Relative Improvement (%)"})

ax.set_title("Pairwise Relative Improvement (%) — Row over Column")
ax.set_xlabel("Baseline (column)")
ax.set_ylabel("Compared (row)")
plt.tight_layout()
plt.show()

---
## 11. Sample Efficiency — Steps to Threshold

How quickly does each ablation reach various score thresholds? Faster is better.

In [ ]:
def steps_to_threshold(records, threshold, key="episode/eval_score", smooth_weight=0.9):
    """Return first step where smoothed eval score exceeds threshold, or NaN."""
    steps, vals = extract_series(records, key)
    if len(steps) == 0:
        return np.nan
    smoothed = smooth(vals, smooth_weight)
    above = np.where(smoothed >= threshold)[0]
    if len(above) == 0:
        return np.nan
    return steps[above[0]]


# Compute thresholds based on a range of the observed scores
all_final = [final_score(all_metrics[ab])[0] for ab in ["A5", "A4"]]
# Use percentiles of the max score for thresholds
max_score = max(s for s in all_final if not np.isnan(s))
thresholds = [max_score * frac for frac in [0.25, 0.5, 0.75]]
thresholds = [round(t / 10) * 10 for t in thresholds]  # round to nearest 10

efficiency_ids = ["A5", "A1", "A3", "A2", "A4", "B3", "B6", "H3"]

rows = []
for ab_id in efficiency_ids:
    row = {"Ablation": ABLATIONS[ab_id][1]}
    for thr in thresholds:
        s = steps_to_threshold(all_metrics[ab_id], thr)
        row[f"Score {thr:.0f}"] = f"{s/1e3:.0f}K" if not np.isnan(s) else "Never"
    rows.append(row)

eff_df = pd.DataFrame(rows).set_index("Ablation")

# Also plot as grouped bar chart
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(efficiency_ids))
width = 0.25

for i, thr in enumerate(thresholds):
    vals = []
    for ab_id in efficiency_ids:
        s = steps_to_threshold(all_metrics[ab_id], thr)
        vals.append(s / 1e3 if not np.isnan(s) else 0)
    bars = ax.bar(x + i * width, vals, width, label=f"Score >= {thr:.0f}",
                  alpha=0.85, edgecolor="black", linewidth=0.5)

ax.set_xticks(x + width)
ax.set_xticklabels([ABLATIONS[ab][1] for ab in efficiency_ids], rotation=20, ha="right")
ax.set_ylabel("Steps to Threshold (K)")
ax.set_title("Sample Efficiency — Steps to Reach Score Thresholds")
ax.legend()
plt.tight_layout()
plt.show()

eff_df

---
## 12. Training Score (Online Episodes) vs Eval Score

Compare online training episode scores with offline evaluation scores to check for overfitting or evaluation-specific effects.

In [ ]:
compare_ids = ["A5", "A4", "H3"]

fig, axes = plt.subplots(1, len(compare_ids), figsize=(6 * len(compare_ids), 5))

for ax, ab_id in zip(axes, compare_ids):
    # Eval score
    steps_e, vals_e = extract_series(all_metrics[ab_id], "episode/eval_score")
    # Train score
    steps_t, vals_t = extract_series(all_metrics[ab_id], "episode/score")

    if len(steps_t) > 0:
        ax.plot(steps_t, smooth(vals_t, 0.95), color="#2ca02c", linewidth=1.5,
                label="Train (online)", alpha=0.8)
        ax.plot(steps_t, vals_t, color="#2ca02c", alpha=0.08, linewidth=0.4)
    if len(steps_e) > 0:
        ax.plot(steps_e, smooth(vals_e, 0.85), color="#1f77b4", linewidth=2.2,
                label="Eval", marker=".", markersize=3)

    ax.set_xlabel("Environment Steps")
    ax.set_ylabel("Score")
    ax.set_title(ABLATIONS[ab_id][1])
    ax.legend(fontsize=9)
    ax.set_xlim(left=0)

plt.suptitle("Training vs. Evaluation Scores", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 13. Comprehensive Radar Chart

Multi-dimensional comparison of ablations across key performance metrics.

In [ ]:
from matplotlib.patches import FancyBboxPatch

radar_ids = ["A5", "A1", "A3", "A2", "A4", "H3"]
radar_colors = [COLORS[ab] for ab in radar_ids]

# Collect metrics for radar
def get_metric_final(records, key, last_n=10):
    steps, vals = extract_series(records, key)
    if len(vals) == 0:
        return np.nan
    n = min(last_n, len(vals))
    return float(np.mean(vals[-n:]))

metric_keys = [
    ("episode/eval_score", "Eval Score"),
    ("train/loss/rew", "Reward Pred\n(lower=better)"),
    ("train/loss/dyn", "Dynamics\n(lower=better)"),
    ("train/ret", "Imagined Return"),
    ("train/action_entropy", "Action Entropy"),
]

# Gather raw values
raw = {ab: [] for ab in radar_ids}
for ab_id in radar_ids:
    for key, _ in metric_keys:
        raw[ab_id].append(get_metric_final(all_metrics[ab_id], key))

# Normalize each dimension to [0, 1]
raw_arr = np.array([raw[ab] for ab in radar_ids])
# For losses, invert so lower = better -> higher normalized value
invert_dims = [1, 2]  # reward and dynamics loss
for d in invert_dims:
    col = raw_arr[:, d]
    valid = ~np.isnan(col)
    if valid.any():
        raw_arr[valid, d] = -col[valid]  # negate so higher is better

# Min-max normalize
normed = np.zeros_like(raw_arr)
for d in range(raw_arr.shape[1]):
    col = raw_arr[:, d]
    valid = ~np.isnan(col)
    if valid.any():
        mn, mx = col[valid].min(), col[valid].max()
        if mx - mn > 1e-8:
            normed[valid, d] = (col[valid] - mn) / (mx - mn)
        else:
            normed[valid, d] = 0.5

# Plot radar
angles = np.linspace(0, 2 * np.pi, len(metric_keys), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for i, ab_id in enumerate(radar_ids):
    vals = normed[i].tolist() + [normed[i][0]]
    ax.plot(angles, vals, color=radar_colors[i], linewidth=2, label=ABLATIONS[ab_id][1])
    ax.fill(angles, vals, color=radar_colors[i], alpha=0.08)

ax.set_thetagrids(np.degrees(angles[:-1]), [name for _, name in metric_keys])
ax.set_ylim(0, 1.1)
ax.set_title("Multi-Metric Radar Comparison", y=1.08, fontsize=14)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

---
## 14. Final Summary Table

Complete results table with all ablations, their architecture, and final scores.

In [ ]:
# Build the full summary table
all_ids_ordered = ["A5", "A1", "A3", "A2", "A4", "B3", "B6",
                   "F1_CNN_med", "F1_CNN_hard", "F1_MM_med", "F1_MM_hard", "H3"]

rows = []
for ab_id in all_ids_ordered:
    mean, std = final_score(all_metrics[ab_id])
    # Improvement over A5 baseline
    a5_mean, _ = final_score(all_metrics["A5"])
    delta = ((mean - a5_mean) / abs(a5_mean) * 100) if (not np.isnan(mean) and not np.isnan(a5_mean) and a5_mean != 0) else np.nan

    row = {
        "ID": ab_id,
        "Label": ABLATIONS[ab_id][1],
        "Group": ABLATIONS[ab_id][2],
        "Mean Score": mean,
        "Std": std,
        "vs. A5 (%)": delta,
    }
    if ab_id in ARCH_META:
        row["FiLM"] = "Yes" if ARCH_META[ab_id]["FiLM"] else "No"
        row["Gate"] = "Yes" if ARCH_META[ab_id]["TextGate"] else "No"
        row["Text"] = ARCH_META[ab_id]["Text"]
    rows.append(row)

final_df = pd.DataFrame(rows)
final_df = final_df.set_index("ID")

# Style it
styled = final_df.style.format({
    "Mean Score": "{:.1f}",
    "Std": "{:.1f}",
    "vs. A5 (%)": "{:+.1f}%"
}).background_gradient(subset=["Mean Score"], cmap="RdYlGn").set_caption(
    "Ablation Results — distract_cheetah_run (1.01M steps)"
)
styled

---
## 15. Export Results

Save all plots and the summary CSV to `ablations/results/`.

In [ ]:
output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)

# Export summary CSV
csv_path = output_dir / "ablation_summary.csv"
final_df.to_csv(csv_path)
print(f"Saved: {csv_path}")

# Print key takeaways
a5_score = final_score(all_metrics["A5"])[0]
a4_score = final_score(all_metrics["A4"])[0]
h3_score = final_score(all_metrics["H3"])[0]

print("\n=== Key Takeaways ===")
if not np.isnan(a4_score) and not np.isnan(a5_score):
    print(f"  Full Multimodal (A4) vs CNN Baseline (A5): {a4_score:.0f} vs {a5_score:.0f} "
          f"({(a4_score - a5_score)/abs(a5_score)*100:+.1f}%)")
if not np.isnan(h3_score) and not np.isnan(a4_score):
    print(f"  Full Multimodal (A4) vs Wider CNN (H3):    {a4_score:.0f} vs {h3_score:.0f} "
          f"({(a4_score - h3_score)/abs(h3_score)*100:+.1f}%)")
    print(f"  -> Advantage is from text semantics, not parameter count")

b3_score = final_score(all_metrics["B3"])[0]
b6_score = final_score(all_metrics["B6"])[0]
if not np.isnan(b3_score) and not np.isnan(b6_score):
    print(f"  Nonsense (B3): {b3_score:.0f}  |  Adversarial (B6): {b6_score:.0f}  |  Real (A4): {a4_score:.0f}")
    print(f"  -> Semantic content {'matters' if a4_score > b3_score else 'may not matter significantly'}")